In [5]:
import sqlalchemy
import torch
from torch.utils.data import DataLoader
import numpy as np
import torch.nn as nn
import torch.optim as optim
from Essentials.Dataset_class import RSNA_Dataset, make_transform
from Essentials.evaluation_loop import Evaluation
from Essentials.training_loop import Training_loop
from Essentials.split_data import Datasplit
from Essentials.models import Densenet_Model, Efficient_Model, Resnet_Model
from sklearn.metrics import classification_report
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight




In [9]:
transform = make_transform()
evaluation = Evaluation()
training = Training_loop()
datasplit = Datasplit()
dense_model = Densenet_Model()
e_model = Efficient_Model()
resnet_model = Resnet_Model()


In [35]:

train , val, test = datasplit.split_data()
dense_train_transform = transform.train_transform(resize=[256], crop_size=[224], mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
dense_val_transform = transform.val_transform(resize=[256], mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
train_dataset = RSNA_Dataset(data=train, transform=dense_train_transform)
val_dataset = RSNA_Dataset(data=val, transform=dense_val_transform)
test_dataset = RSNA_Dataset(data=test, transform=dense_val_transform)
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=64)
val_dataloader = DataLoader(val_dataset, shuffle=False, batch_size=64)
test_dataloader = DataLoader(test_dataset, shuffle=False, batch_size=64)


In [4]:
dense_model_classifier = dense_model.classifier_only()
optimization = optim.Adam(dense_model_classifier.classifier.parameters(), lr=0.0001)
lr_sh = optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimization, patience=2, factor=0.1, mode='max')
device = torch.device('mps' if torch.mps.is_available() else 'cpu')
weights = compute_class_weight(class_weight='balanced', classes=np.array([0, 1]), y=np.array(train['Target']))
weights = (torch.tensor(weights, dtype=torch.float)).to(device)
loss_function = nn.CrossEntropyLoss()


In [19]:

#----------------------Training Loop -------------------

dense_model_classifier = training.training_loop(model=dense_model_classifier, loss_function=loss_function, optimization=optimization, lr_sch=lr_sh, model_name='dense',train_dataloader=train_dataloader, val_dataloader=val_dataloader)

In [10]:
dense_model_classifier = dense_model.load_model(classifier_only=True)
val_loss, f1, real, predicted = evaluation.evaluate(dense_model_classifier, loss_function=loss_function, val_dataloader=val_dataloader, train=False)

In [11]:
dense_classifier_report = classification_report(real, predicted)
print(dense_classifier_report)

              precision    recall  f1-score   support

           0       0.91      0.73      0.81      5027
           1       0.57      0.83      0.68      2171

    accuracy                           0.76      7198
   macro avg       0.74      0.78      0.74      7198
weighted avg       0.81      0.76      0.77      7198



In [20]:
#----------------------One Unfreezed Layer-------------------
dense_model_unfreezed = dense_model.one_unfreezed_layer()

In [19]:
#----------------------Training Loop -------------------
retrained_optimizer = optim.Adam(dense_model.parameters(), lr=0.00001)
dense_model_unfreezed = training.training_loop(model=dense_model, loss_function=loss_function, optimization=retrained_optimizer, model_name='retrained_dense', train_dataloader=train_dataloader, lr_sch=lr_sh, val_dataloader=val_dataloader)

In [13]:
dense_model_unfreezed = dense_model.load_model(classifier_only=False)
val_loss, f1, real, predicted = evaluation.evaluate(dense_model_unfreezed, loss_function=loss_function, val_dataloader=val_dataloader, train=False)

In [14]:
dense_unfreezed_report = classification_report(real, predicted)
print(dense_unfreezed_report)

              precision    recall  f1-score   support

           0       0.93      0.82      0.87      5027
           1       0.67      0.86      0.75      2171

    accuracy                           0.83      7198
   macro avg       0.80      0.84      0.81      7198
weighted avg       0.85      0.83      0.83      7198



Efficient Net model


In [10]:
emodel_train_transform = transform.train_transform(resize=[288], crop_size=[288], mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
emodel_val_transform = transform.val_transform(resize=[288], mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
train_dataset = RSNA_Dataset(data=train, transform=emodel_train_transform)
val_dataset = RSNA_Dataset(data=val, transform=emodel_val_transform)
test_dataset = RSNA_Dataset(data=test, transform=emodel_val_transform)
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=64)
val_dataloader = DataLoader(val_dataset, shuffle=False, batch_size=64)
test_dataloader = DataLoader(test_dataset, shuffle=False, batch_size=64)


In [19]:
#----------------------Training Loop -------------------
e_model_classifier  = e_model.classifier_only()
optimization = optim.Adam(e_model_classifier.classifier.parameters(), lr=0.0001)
e_model_classifier = training.training_loop(e_model_classifier, loss_function=loss_function, optimization=optimization, lr_sch=lr_sh, model_name='efficient_model_classifier', train_dataloader=train_dataloader, val_dataloader=val_dataloader)


In [11]:
e_model_classifier = e_model.load_model(classifier_only=True)
val_loss, f1, real, predicted = evaluation.evaluate(e_model_classifier, loss_function=loss_function, val_dataloader=val_dataloader, train=False)


In [12]:
efficient_classifier_report = classification_report(real, predicted)
print(efficient_classifier_report)

              precision    recall  f1-score   support

           0       0.90      0.76      0.82      4920
           1       0.62      0.82      0.70      2334

    accuracy                           0.78      7254
   macro avg       0.76      0.79      0.76      7254
weighted avg       0.81      0.78      0.78      7254



Efficient Net Model Unfreezed

In [19]:
e_model_unfreezed = e_model.one_unfreezed_layer()
retrained_optimizer_e = optim.Adam(e_model_unfreezed.parameters(), lr=0.00001)
e_model_unfreezed = training.training_loop(model=e_model_unfreezed, optimization=retrained_optimizer_e, loss_function=loss_function, val_dataloader=val_dataloader, train_dataloader=train_dataloader, lr_sch=lr_sh, model_name='e_model_unfreezed')

In [13]:
e_model_unfreezed = e_model.load_model(classifier_only=False)
val_loss, f1, real, predicted = evaluation.evaluate(e_model_unfreezed, loss_function=loss_function, val_dataloader=val_dataloader, train=False)



In [14]:
efficient_unfreezed_report = classification_report(real, predicted)
print(efficient_unfreezed_report)

              precision    recall  f1-score   support

           0       0.90      0.76      0.82      4920
           1       0.62      0.83      0.71      2334

    accuracy                           0.78      7254
   macro avg       0.76      0.79      0.77      7254
weighted avg       0.81      0.78      0.79      7254



Resnet Model

In [11]:

transform_resent = transform.train_transform(resize=[256], crop_size=[224], mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
train_dataset = RSNA_Dataset(train, transform=transform_resent)
val_transform_resnet = transform.val_transform(resize=[256], mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
val_dataset = RSNA_Dataset(val, transform=val_transform_resnet )
test_dataset = RSNA_Dataset(test, transform=val_transform_resnet )
train_dataloader = DataLoader(batch_size=64, dataset=train_dataset, shuffle=True)
val_dataloader = DataLoader(batch_size=64, dataset=val_dataset, shuffle=False)
test_dataloader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)


In [19]:
#----------------------Training Loop -------------------
resnet_classifier = resnet_model.classifier_only()
optimization = optim.Adam(resnet_classifier.parameters(), lr=0.0001)
resnet_classifier = training.training_loop(model = resnet_classifier, optimization=optimization, loss_function=loss_function, train_dataloader=train_dataloader, lr_sch=lr_sh, val_dataloader=val_dataloader, model_name='resnet_classifier')


In [6]:
resnet_classifier = resnet_model.load_model(classifier_only=True)
val_loss, f1, real, predicted = evaluation.evaluate(resnet_classifier, loss_function=loss_function, val_dataloader=val_dataloader, train=False)

In [7]:
resnet_classifier_report = classification_report(real, predicted)
print(resnet_classifier_report)

              precision    recall  f1-score   support

           0       0.84      0.86      0.85      5027
           1       0.65      0.61      0.63      2171

    accuracy                           0.78      7198
   macro avg       0.74      0.74      0.74      7198
weighted avg       0.78      0.78      0.78      7198



Unfreezed Resnet Model

In [19]:
resnet_unfreezed = resnet_model.classifier_only()
retrained_optimizer_resnet = optim.Adam(resnet_unfreezed.parameters(), lr=0.00001)
resnet_unfreezed = training.training_loop(model=resnet_unfreezed, optimization=retrained_optimizer_resnet, loss_function=loss_function, val_dataloader=val_dataloader, train_dataloader=train_dataloader, lr_sch=lr_sh, model_name='resnet_unfreezed')


In [12]:
resnet_unfreezed = resnet_model.load_model(classifier_only=False)

In [8]:

val_loss, f1, real, predicted = evaluation.evaluate(resnet_unfreezed, loss_function=loss_function, val_dataloader=val_dataloader, train=False)

In [9]:
resnet_unfreezed_report = classification_report(real, predicted)
print(resnet_unfreezed_report)

              precision    recall  f1-score   support

           0       0.92      0.86      0.89      5027
           1       0.72      0.83      0.77      2171

    accuracy                           0.85      7198
   macro avg       0.82      0.84      0.83      7198
weighted avg       0.86      0.85      0.85      7198



In [15]:
#---------------------Test Set---------------------
loss, f1, real, predicted = evaluation.evaluate(resnet_unfreezed, loss_function=loss_function, val_dataloader=test_dataloader, train=False)



In [16]:
#---------------------Test Set Result---------------------
print(classification_report(real, predicted))

              precision    recall  f1-score   support

           0       0.92      0.87      0.89      1245
           1       0.75      0.83      0.79       591

    accuracy                           0.86      1836
   macro avg       0.83      0.85      0.84      1836
weighted avg       0.86      0.86      0.86      1836



In [13]:

resnet_unfreezed.eval()
# with torch.no_grad():

data = pd.read_csv('UserCase_studies.csv')

In [14]:
patient_ids = data.iloc[:,0][0]
patient_ids

'2fcba0c2-b470-4d23-8701-f4d2fdfd700f'

In [15]:
import pydicom
test_image = pydicom.dcmread(f'Dataset/rsna-pneumonia-detection-challenge/stage_2_test_images/{patient_ids}.dcm')

In [30]:
import torch
device = torch.device('mps' if torch.mps.is_available() else 'cpu')
transformed_test = val_transform_resnet(test_image.pixel_array)
resnet_unfreezed = resnet_unfreezed.to(device)
with torch.no_grad():
    test = transformed_test.unsqueeze(0).to(device)
    logits = resnet_unfreezed(test)
    prob = torch.softmax(logits, dim=1)
    print(round(torch.max(prob).item(), 2))



0.58


In [83]:
type(test_image.pixel_array)

numpy.ndarray

In [33]:
resnet_unfreezed.eval()
p = []
a = []
with torch.no_grad():
    for image, label in val_dataloader:
        image, label = image.to(device), label.to(device)
        predictions = resnet_unfreezed(image)
        p.extend(torch.softmax(predictions, dim=1).tolist())
        a.extend(label.tolist())
p = torch.tensor(p)
disease = p[:, 1]
pnuenomia = torch.where(disease.clone()>=0.45, 1, 0)
import numpy as np
from sklearn.metrics import precision_recall_curve
precision , recall , thresholds = precision_recall_curve(a, disease)

f1_scores = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-10)
)

best_index = np.argmax(f1_scores)

best_threshold = thresholds[best_index]

print("Best threshold:", best_threshold)
print("Best F1:", f1_scores[best_index])
print("Precision:", precision[best_index])
print("Recall:", recall[best_index])

Best threshold: 0.5296481
Best F1: 0.7922265866773671
Precision: 0.7664
Recall: 0.819854514334617


In [1]:
import torch
import torchvision
print(torch.__version__)

2.13.0


In [2]:
print(torchvision.__version__)

0.28.0


In [2]:
import psycopg2
print(psycopg2.__version__)

2.9.12 (dt dec pq3 ext lo64)


In [ ]:
import pandas as pd
pd.read_csv('Essentials/resnet_model/unfreezedresnet_bestweights.pth')

In [1]:
python-multipart
